# Phase II — Interactive Detective Mystery (Colab demo)

Runs the full Template 2 (Intervention & Accommodation) system entirely inside Colab.

### Requirements

- **Runtime → Change runtime type → GPU** (any of T4 / L4 / A100 works).
- Free-tier T4 (16 GB VRAM) can host `Qwen/Qwen2.5-3B-Instruct` comfortably. Select a bigger model only if you have L4 / A100.
- No Anthropic account needed. The entire stack runs on a local vLLM OpenAI-compatible server inside this notebook.

### What this notebook does

1. Installs vLLM + openai client.
2. Clones the repo.
3. Starts a local vLLM server in the background on `localhost:8000`.
4. Smoke-tests connectivity via `scripts/test_llm.py`.
5. Generates **one fixed** story instance (`data/plan.json`, `data/world.json`).
6. Replays two scripted playthroughs:
   - `success_run.txt` — detective solves the case.
   - `exception_run.txt` — detective destroys evidence, drama manager accommodates.
7. Prints the last 20 lines of `logs/drama.jsonl` so you can see the classifications and plan repairs.

## 1. Install dependencies

vLLM ships with its own torch+CUDA wheels. The install takes ~3-5 min on Colab.

In [ ]:
!pip install --quiet 'vllm>=0.11,<0.13' 'openai>=1.52'
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('GPU memory:', f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else 'n/a')

## 2. Clone the repo

Replace the URL if you're working from a fork.

In [ ]:
import os, subprocess, pathlib
REPO_URL = 'https://github.com/ivy3h/7634-s26.git'
WORKDIR  = '/content/7634-s26'
if not pathlib.Path(WORKDIR).exists():
    subprocess.check_call(['git', 'clone', REPO_URL, WORKDIR])
else:
    subprocess.check_call(['git', '-C', WORKDIR, 'pull', '--ff-only'])
os.chdir(WORKDIR)
print('cwd =', os.getcwd())
!ls

## 3. Configure model + endpoint

Pick a model that fits your Colab GPU:

| GPU VRAM | Recommended model | Weights (bf16) |
|---|---|---|
| 16 GB or less (T4) | `Qwen/Qwen2.5-3B-Instruct` | ~6 GB |
| 24 GB (L4)         | `Qwen/Qwen2.5-7B-Instruct` | ~14 GB |
| 40 GB (A100)       | `Qwen/Qwen3-8B`            | ~16 GB |

In [ ]:
import os, torch

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if gpu_gb >= 35:
    MODEL = 'Qwen/Qwen3-8B'
elif gpu_gb >= 22:
    MODEL = 'Qwen/Qwen2.5-7B-Instruct'
else:
    MODEL = 'Qwen/Qwen2.5-3B-Instruct'

PORT = 8000
os.environ['LLM_MODEL'] = MODEL
os.environ['LLM_ENDPOINT'] = f'http://localhost:{PORT}/v1'
os.environ['LLM_API_KEY'] = 'EMPTY'
print('Model    :', MODEL)
print('Endpoint :', os.environ['LLM_ENDPOINT'])

## 4. Launch vLLM server in the background

First run on a fresh model takes ~3-8 min (download + load).

In [ ]:
import subprocess, time, pathlib, sys
pathlib.Path('logs').mkdir(exist_ok=True)
log_path = pathlib.Path('logs/vllm_server.log')
log_path.write_text('')

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--host', '0.0.0.0', '--port', str(PORT),
    '--dtype', 'bfloat16',
    '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--api-key', 'EMPTY',
]
print('Launching:', ' '.join(cmd))
server = subprocess.Popen(cmd, stdout=open(log_path, 'ab'), stderr=subprocess.STDOUT)
print('pid =', server.pid)
pathlib.Path('logs/vllm_endpoint.txt').write_text(os.environ['LLM_ENDPOINT'])

## 5. Wait for the server to be ready

In [ ]:
import time, urllib.request, json

def probe():
    try:
        req = urllib.request.Request(f'{os.environ["LLM_ENDPOINT"]}/models', headers={'Authorization': 'Bearer EMPTY'})
        with urllib.request.urlopen(req, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

for i in range(240):  # up to ~20 min
    if probe():
        print('READY after', i*5, 's')
        break
    if i % 12 == 0:
        print('... still waiting (', i*5, 's)', flush=True)
    time.sleep(5)
else:
    raise RuntimeError('vLLM did not become ready in time - check logs/vllm_server.log')

## 6. Smoke test

In [ ]:
!python scripts/test_llm.py

## 7. Build the story + plan + world

This runs Phase I story generation, then story_to_plan, then world_builder, against the live server. Total ~3-10 min depending on model size. The result is saved to `data/` so subsequent playthroughs reuse the same fixed story.

In [ ]:
!python main.py build --data-dir data --min-points 15

## 8. Replay a successful run

In [ ]:
!python main.py replay transcripts/success_run.txt --data-dir data --log-dir logs

## 9. Replay an exception run (drama manager accommodates)

In [ ]:
!python main.py replay transcripts/exception_run.txt --data-dir data --log-dir logs

## 10. Inspect the drama manager log

Every behind-the-scenes decision - classifications, commonsense threat checks, removed events, replacement events - is in `logs/drama.jsonl`. Each line is one JSON object.

In [ ]:
import json, pathlib
for line in pathlib.Path('logs/drama.jsonl').read_text().splitlines()[-20:]:
    entry = json.loads(line)
    kind = entry.pop('kind')
    print(kind, '-', json.dumps(entry)[:200])

## 11. (Optional) Interactive mode

Colab cells can't accept live stdin elegantly, but you can drive the engine programmatically by importing `GameEngine` and passing a Python callable as `get_input`.

In [ ]:
from game_engine import GameEngine, EngineConfig
from story_to_plan import load_plan
from world_builder import load_world

plan = load_plan('data/plan.json')
world = load_world('data/world.json')
engine = GameEngine(plan, world, EngineConfig(log_dir=pathlib.Path('logs/interactive')))

def ask(prompt):
    # Swap this for `input(prompt)` if Colab stdin works for you,
    # or plug in your own policy.
    return input(prompt)

status = engine.run(get_input=ask, echo=print)
print('\n=== game ended:', status, '===')

## 12. Cleanup

Kill the background server before ending the session (frees the GPU for the next notebook). Running this cell is optional - Colab will kill it when the runtime disconnects.

In [ ]:
try:
    server.terminate()
    server.wait(timeout=10)
    print('server stopped')
except Exception as e:
    print('cleanup:', e)